# Parameter-shift gradients

Differentiate the same QNode with the hardware-compatible parameter-shift rule on both devices.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

Parameter-shift differentiation evaluates shifted circuits and combines their expectation values into a gradient.

In [2]:
def make_qnode(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(theta):
        qml.RX(theta, wires=0)
        qml.RY(-0.23, wires=1)
        qml.CNOT(wires=[0, 1])
        return qml.expval(qml.Z(1))
    return circuit

theta = pnp.array(0.41, requires_grad=True)
reference_qnode = make_qnode(qml.device("default.qubit", wires=2))
def reference_value_gradient():
    return float(reference_qnode(theta)), float(qml.grad(reference_qnode)(theta))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(reference_value_gradient)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
def mettleq_value_gradient():
    return float(mettleq_qnode(theta)), float(qml.grad(mettleq_qnode)(theta))
candidate, mettleq_ms, _ = benchmark(mettleq_value_gradient)
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

Both the forward value and gradient must agree with default.qubit.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/03_parameter_shift_gradients.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="value and parameter-shift gradient atol=3e-6",
    passed=error <= 3e-6,
    exact_match=reference == candidate,
    selected_method=method,
    selected_device=device,
    metrics={"max_value_or_gradient_error": error, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — value and parameter-shift gradient atol=3e-6
SDK reference median: 2.828 ms
MettleQ median:       8.030 ms
Timing interpretation: the SDK reference was 2.839x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "value and parameter-shift gradient atol=3e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_value_or_gradient_error": 7.084163977655322e-09, "mettleq": [0.8929697275161743, -0.38811250030994426], "reference": [0.892969725336207, -0.38811250739410824]}, "mettleq_median_ms": 8.029583987081423, "notebook": "pennylane/03_parameter_shift_gradients.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 2.8279999969527125, "reference_over_mettleq": 0.35219757356079767, "schema_version": 1, "s

## What should you conclude?

Small gradients are call-overhead bound. Larger parameter batches are the meaningful acceleration target.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.